In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
import torchtext; torchtext.disable_torchtext_deprecation_warning()
from torchtext.data.utils import get_tokenizer
import os
from dataset import clean_data, create_text_data, split_data, get_transforms
from inference import inference
from PIL import Image
from model import EncoderCNN
import torch

In [ ]:
# init directory and file path
root_path = '../'
data_root_path = f"{root_path}dataset"
text_dataset = f"{data_root_path}/text/Flickr8k.token.bahasa.txt"
clean_text_dataset = f"{data_root_path}/text/Flickr8k.token.bahasa.clean.txt"

train_file = data_root_path + "/text/" + "Flickr_8k.trainImages.txt"
val_file = data_root_path +  "/text/" + "Flickr_8k.devImages.txt"
test_file = data_root_path + "/text/" + "Flickr_8k.testImages.txt"

# init tokenizer
tokenizer = get_tokenizer('basic_english')

# Clean dataset
if not os.path.exists(clean_text_dataset):
    clean_data(text_dataset, clean_text_dataset, data_root_path)
    
dataframe, vocab = create_text_data(clean_text_dataset, tokenizer=tokenizer)

# split data train, val, test
_, _, test_df = split_data(dataframe, train_file, val_file, test_file)

In [ ]:
d_model = 512
n_head = 4
cnn_size = 'm'
model_path = f'saved_model/{d_model}_{n_head}_{cnn_size}.pth'
image_size = 288
transforms = get_transforms(image_size)
cnn_backbone = 'efficientnetv2_rw_s.ra2_in1k'
cnn_backbone = 'efficientnetv2_rw_m.agc_in1k'

In [ ]:
# # Setup
# batch_size = 32  # or any other size suitable for your hardware
# data = {
#     'filename': [],
#     'predicted': [],
#     'target': []
# }

# # Process in batches
# for i in range(0, len(test_df), batch_size):
#     batch_filenames = test_df['image_filename'][i:i+batch_size]
#     batch_captions = test_df['captions'][i:i+batch_size]
#     batch_images = [Image.open(f"{data_root_path}/images/{filename}").convert('RGB') for filename in batch_filenames]

#     # Run batch inference
#     batch_predictions = batch_inference(batch_images, model_path, d_model, n_head, vocab)

#     # Process each prediction in the batch
#     for filename, captions, prediction_words in zip(batch_filenames, batch_captions, batch_predictions):
#         predicted = ' '.join(word for word in prediction_words if word != '<end>')
#         data['filename'].append(filename)
#         data['predicted'].append(predicted)
#         data['target'].append(captions)

In [ ]:
print(cnn_backbone)

In [ ]:
import timm
import torchvision.models as models
from torchsummary import summary

cnn_backbone = 'efficientnetv2_rw_m.agc_in1k'
cnn_backbone = 'efficientnetv2_rw_s.ra2_in1k'

device = 'cuda'
model = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT)
# model = timm.create_model(cnn_backbone, pretrained=True, features_only=True)
model.to(device)
# modules = model._modules
# list(modules)
# modules['features']

# print("Modules in 'features':")
# for name, module in model.features.named_children():
#     print(module)
#     print("Length:", len(module))
#     print("\n")
# summary(model, input_size=(3, 384, 384))

In [71]:
from urllib.request import urlopen
from PIL import Image
import timm

img = Image.open(urlopen(
    'https://images.unsplash.com/photo-1493612276216-ee3925520721?q=80&w=1000&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxzZWFyY2h8Mnx8cmFuZG9tfGVufDB8fDB8fHwwg'
))

cnn_backbone = 'efficientnetv2_rw_s.ra2_in1k'
cnn_backbone = 'efficientnetv2_rw_m.agc_in1k'

model = timm.create_model(
    cnn_backbone,
    pretrained=True,
    features_only=True,
)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)
print(data_config)

output = model(transforms(img).unsqueeze(0))  # unsqueeze single image into batch of 1
print(output[-1].shape)

Unexpected keys (bn2.bias, bn2.num_batches_tracked, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


{'input_size': 416, 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 1.0, 'crop_mode': 'center'}
torch.Size([1, 328, 13, 13])


In [ ]:
import torch
# Determine the input channel of the linear layer
image_size = 384
dummy_input = torch.randn((1, 3, image_size, image_size)).to(device)
model.eval()
with torch.no_grad():
    features = model (dummy_input)
print(features.shape)

In [ ]:
import pandas as pd
from tqdm import tqdm

data = {
    'filename': [],
    'predicted': [],
    'target': []
}
start = 1
end = 20
# Use tqdm to wrap your iterable
for image_filename, captions in tqdm(zip(test_df['image_filename'][start:end], test_df['captions'][start:end]), total=end-start):
    image = Image.open(f"{data_root_path}/images/{image_filename}").convert('RGB')
    
    words = inference(image, model_path, d_model, n_head, vocab, cnn_backbone)
    predicted = ' '.join(word for word in words if word != '<end>')
    new_row = {
        'filename' : image_filename,
        'predicted': predicted,
        'target': captions  # Assuming you want to collect captions as target
    }
    
    for key, value in new_row.items():
        data[key].append(value)
    
data_df = pd.DataFrame(data)
data_df.to_csv(f'testing_greedy_{start}_{end}_{cnn_size}.csv', sep=",", index_label='index')

In [ ]:
import pandas as pd
from tqdm import tqdm

data = {
    'filename': [],
    'predicted': [],
    'target': []
}
start = 500
end = 750
# Use tqdm to wrap your iterable
for image_filename, captions in tqdm(zip(test_df['image_filename'][start:end], test_df['captions'][start:end]), total=end-start):
    image = Image.open(f"{data_root_path}/images/{image_filename}").convert('RGB')
    
    words = inference(image, model_path, d_model, n_head, vocab, cnn_backbone)
    predicted = ' '.join(word for word in words if word != '<end>')
    new_row = {
        'filename' : image_filename,
        'predicted': predicted,
        'target': captions  # Assuming you want to collect captions as target
    }
    
    for key, value in new_row.items():
        data[key].append(value)
    
data_df = pd.DataFrame(data)
data_df.to_csv(f'testing_results/testing_{start}_{end}_{cnn_size}.csv', sep=",", index_label='index')